Day3: 按 category 挖 “高频短语/关键词”，再半自动生成“正向卖点词/负向风险词”候选

In [3]:
!pip -q install scikit-learn

In [4]:
import pandas as pd
from pathlib import Path

OUT = Path("../data_processed")

reviews = pd.read_csv(OUT / "reviews.csv")
products = pd.read_csv(OUT / "product_facts.csv")

print("reviews:", reviews.shape, reviews.columns.tolist())
print("products:", products.shape, products.columns.tolist())

reviews.head(2), products.head(2)

reviews: (1093667, 4) ['product_id', 'rating', 'review_text', 'review_date']
products: (8494, 13) ['product_id', 'product_name', 'brand_name', 'primary_category', 'secondary_category', 'tertiary_category', 'price_usd', 'sale_price_usd', 'ingredients', 'highlights', 'rating', 'review_cnt', 'rating_avg']


(  product_id  rating                                        review_text  \
 0    P504322       5  I use this with the Nudestix “Citrus Clean Bal...   
 1    P420652       1  I bought this lip mask after reading the revie...   
 
   review_date  
 0  2023-02-01  
 1  2023-03-21  ,
   product_id             product_name brand_name primary_category  \
 0    P473671  Fragrance Discovery Set      19-69        Fragrance   
 1    P473668  La Habana Eau de Parfum      19-69        Fragrance   
 
   secondary_category  tertiary_category  price_usd  sale_price_usd  \
 0  Value & Gift Sets  Perfume Gift Sets       35.0            35.0   
 1              Women            Perfume      195.0           195.0   
 
                                          ingredients  \
 0  ['Capri Eau de Parfum:', 'Alcohol Denat. (SD A...   
 1  ['Alcohol Denat. (SD Alcohol 39C), Parfum (Fra...   
 
                                           highlights  rating  review_cnt  \
 0  ['Unisex/ Genderless Scent', 'Warm &S

In [5]:
df = reviews.merge(
    products[["product_id", "primary_category"]],
    on="product_id",
    how="left"
)

print("merged:", df.shape)
print("missing primary_category:", df["primary_category"].isna().mean())
df.head(2)

merged: (1093667, 5)
missing primary_category: 0.0


,product_id,rating,review_text,review_date,primary_category
0,P504322,5,I use this with the Nudestix “Citrus Clean Bal...,2023-02-01,Skincare
1,P420652,1,I bought this lip mask after reading the revie...,2023-03-21,Skincare


In [6]:
df["review_text"] = df["review_text"].astype(str)
df["review_text_clean"] = (
    df["review_text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df[df["review_text_clean"].str.len() >= 20]
df = df.dropna(subset=["primary_category"])

print("after clean:", df.shape)

after clean: (1092309, 6)


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

def top_ngrams_tfidf(texts, top_k=30, ngram_range=(1,2), min_df=5, max_df=0.8):
    vec = TfidfVectorizer(
        stop_words="english",
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df
    )
    X = vec.fit_transform(texts)
    terms = np.array(vec.get_feature_names_out())
    scores = np.asarray(X.mean(axis=0)).ravel()  # 平均 tf-idf
    top_idx = scores.argsort()[::-1][:top_k]
    return list(zip(terms[top_idx], scores[top_idx]))

# 先挑一个类目看看
one_cat = df["primary_category"].value_counts().index[0]
samples = df.loc[df["primary_category"] == one_cat, "review_text_clean"].sample(
    min(20000, (df["primary_category"] == one_cat).sum()), random_state=42
)

print("category:", one_cat, "n_texts:", len(samples))
top_ngrams_tfidf(samples, top_k=30)

category: Skincare n_texts: 20000


[('skin', np.float64(0.045071262204387265)),
 ('product', np.float64(0.030641693292070043)),
 ('love', np.float64(0.024937005388699796)),
 ('use', np.float64(0.02049926755818018)),
 ('like', np.float64(0.02004469896416911)),
 ('face', np.float64(0.019890770054298297)),
 ('really', np.float64(0.017724626416509674)),
 ('using', np.float64(0.017635534847378797)),
 ('great', np.float64(0.016229213041100435)),
 ('dry', np.float64(0.015013141532871428)),
 ('feel', np.float64(0.014131237934678326)),
 ('ve', np.float64(0.013641623622484741)),
 ('just', np.float64(0.013115730110616036)),
 ('moisturizer', np.float64(0.0127303947255703)),
 ('cream', np.float64(0.012716355565432718)),
 ('good', np.float64(0.01245421733615931)),
 ('used', np.float64(0.012445700640111804)),
 ('makeup', np.float64(0.012178473675116973)),
 ('feels', np.float64(0.012116142764903201)),
 ('definitely', np.float64(0.011403947368636891)),
 ('amazing', np.float64(0.010935173560865097)),
 ('recommend', np.float64(0.010903416

In [8]:
cats = df["primary_category"].value_counts()
cats = cats[cats >= 200]  # 过滤太小类目
cats.index.tolist()[:10], len(cats)

(['Skincare'], 1)

In [9]:
rows = []
for cat in cats.index:
    sub = df.loc[df["primary_category"] == cat, "review_text_clean"]
    sub = sub.sample(min(20000, len(sub)), random_state=42)

    top_terms = top_ngrams_tfidf(sub, top_k=80, ngram_range=(1,2), min_df=5, max_df=0.8)
    for term, score in top_terms:
        rows.append({"category": cat, "keyword": term, "tfidf": float(score)})

kw = pd.DataFrame(rows)
print("kw shape:", kw.shape)
kw.head()

kw shape: (80, 3)


,category,keyword,tfidf
0,Skincare,skin,0.045071
1,Skincare,product,0.030642
2,Skincare,love,0.024937
3,Skincare,use,0.020499
4,Skincare,like,0.020045
